# Qwen3.8 Flash-Next → existing DeepSeek Harness Qwen slot

Drop-in replacement for `Qwen3_8_27B_API_Colab.ipynb`. Windows/Harness stays unchanged: `http://127.0.0.1:8787/v1`, model id `qwen3.8-27b`, relay id `qwen3-8-27b`, same bridge/settings/affinity.

Primary target: **Colab G4 = NVIDIA RTX PRO 6000 Blackwell Server Edition (~95.6 GiB VRAM + ~176.9 GiB system RAM)**. The runtime uses native FP8 for complete GPU-authoritative expert banks and adaptively splits the remaining expert banks into host RAM with a dynamic global GPU cache. GPU-authoritative banks do **not** retain duplicate CPU copies. The giant PLE/n-gram table stays host-resident.

## Section 1 — Install / update Flash worker
Run once in a fresh runtime. The planner uses the actual RAM/VRAM it detects. The G4 96 GB / ~177 GiB RAM configuration is the intended target. Roughly 190 GiB free local storage is still required for the real FP8 checkpoint.

In [ ]:
!rm -rf /content/All-testing
!git clone --depth 1 --branch qwen38-flash-freetoken-colab https://github.com/Logan17de/All-testing.git /content/All-testing
%cd /content/All-testing/llm
%pip install -q -U -r qwen38_flash_freetoken/requirements-colab.txt
%pip install -q -U -e ".[qwen38-flash-freetoken]"
import importlib.metadata as metadata
print(f"all-testing-llm {metadata.version('all-testing-llm')}: OK ✅")

## Section 2A — EXPECTED-OOM real production-path smoke test
This section intentionally ends in CUDA/VRAM OOM; that OOM is the PASS condition. Before it can pass, it validates the real checkpoint, relay, live architecture, **adaptive CPU/GPU authoritative expert-layer plan**, native Blackwell FP8 path, custom expert backend, real model load, dynamic expert cache, measured bandwidth and a real routed generation.

If the real runtime hits a natural CUDA memory wall, that counts as PASS. If the model and generation succeed, the test deliberately allocates beyond remaining VRAM to produce the terminal expected OOM. Any non-memory exception before OOM is a real failure to fix.

In [ ]:
import qwen3_8_flash_supabase_colab_oom_smoke as qwen_oom_smoke
qwen_oom_smoke.main()

## Section 2B — REAL Qwen3.8 Flash Harness worker
Run after Section 2A reaches `EXPECTED VRAM LIMIT REACHED ✅ — SMOKE TEST PASSED`. It uses the same adaptive placement plan, starts the private local OpenAI-compatible server in Colab, and attaches it to the same Supabase relay used by the old Qwen3.8-27B worker. No Windows/Harness change is required.

In [ ]:
import qwen3_8_flash_colab_runtime as qwen_worker
qwen_worker.main()

## Section 3 — TESTING: existing Harness relay only (NO GPU/model)
Use a CPU Colab runtime only to prove the existing Windows Harness → Supabase → Colab transport. Run Section 1 and this cell instead of Sections 2A/2B. Every valid request returns `succeed`.

In [ ]:
import qwen_supabase_test_worker as relay_test
relay_test.main()

## Normal startup
1. Stop the old Qwen3.8-27B Colab worker.
2. Run Section 1.
3. Run Section 2A; fix every non-memory error until expected-OOM PASS.
4. Run Section 2B and leave it running.
5. Keep using the same Windows bridge and `qwen3.8-27b` Harness model entry.